In [2]:
print("a")

a


In [3]:
from pathlib import Path
import requests

from pystac_client import Client
import planetary_computer

OUT_DIR = Path("../datasets/sentinel2_visual")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:

# 例: 東京湾のざっくり bbox [min_lon, min_lat, max_lon, max_lat]
bbox = [139.5, 35.2, 140.2, 35.9]

catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2024-01-01/2025-12-31",
    query={"eo:cloud_cover": {"lt": 20}},
)

items = list(search.items())

print(f"found: {len(items)} items")

for item in items[:100]:
    asset = item.assets.get("visual")
    if asset is None:
        continue

    url = asset.href
    out = OUT_DIR / f"{item.id}_visual.tif"
    if out.exists():
        continue

    print("downloading", out.name)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(out, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

found: 264 items
downloading S2B_MSIL2A_20251207T012939_R074_T54SVD_20251207T030103_visual.tif
downloading S2B_MSIL2A_20251207T012939_R074_T54SUE_20251207T030103_visual.tif
downloading S2B_MSIL2A_20251207T012939_R074_T54SUD_20251207T030103_visual.tif
downloading S2A_MSIL2A_20251204T013111_R074_T54SVE_20251204T043021_visual.tif
downloading S2A_MSIL2A_20251204T013111_R074_T54SUE_20251204T043021_visual.tif
downloading S2A_MSIL2A_20251204T013111_R074_T54SUD_20251204T043021_visual.tif
downloading S2B_MSIL2A_20251204T011929_R031_T54SVE_20251204T024635_visual.tif
downloading S2C_MSIL2A_20251202T013031_R074_T54SVD_20251202T043310_visual.tif
downloading S2C_MSIL2A_20251202T013031_R074_T54SUD_20251202T043310_visual.tif
downloading S2A_MSIL2A_20251201T012111_R031_T54SVE_20251201T041915_visual.tif
downloading S2A_MSIL2A_20251201T012111_R031_T54SVD_20251201T041915_visual.tif
downloading S2B_MSIL2A_20251127T012909_R074_T54SVE_20251127T025419_visual.tif


HTTPError: 403 Client Error: Server failed to authenticate the request. Make sure the value of Authorization header is formed correctly including the signature. for url: https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/54/S/VE/2025/11/27/S2B_MSIL2A_20251127T012909_N0511_R074_T54SVE_20251127T025419.SAFE/GRANULE/L2A_T54SVE_A045574_20251127T012908/IMG_DATA/R10m/T54SVE_20251127T012909_TCI_10m.tif?st=2026-03-21T08%3A27%3A59Z&se=2026-03-22T09%3A12%3A59Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-03-22T07%3A30%3A04Z&ske=2026-03-29T07%3A30%3A04Z&sks=b&skv=2025-07-05&sig=FOSrW4kXy3WLSPuXEFE8gnneO9GatqrJELTLTGOmbdU%3D